In [ ]:
from pathlib import Path
import os
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Python: {sys.executable}')
print(f'Project root: {project_root}')
print(f'Data folder exists: {(project_root / "data").exists()}')

: 

### Neural Network for WMH Segmentation

Pipeline: 
- build a pixel-wise DataFrame
- handle class imbalance
- train a neural network
- plot learning curves
- evaluate with confusion matrices

In [ ]:
from pathlib import Path
import os
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")
os.environ["MPLCONFIGDIR"] = str(project_root / ".matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from wmh_preprocessing import load_dataset, prepare_slice_features

data_dir = project_root / "data"
output_dir = project_root / "nn_output"
output_dir.mkdir(exist_ok=True)


feature_columns = [
    "intensity",
    "mean",
    "std",
    "skewness",
    "kurtosis",
    "contrast",
    "range",
    "x",
    "y",
]

###  Create the DataFrame 

Each row is one pixel, each column is one features. `wmh` is the binary target label.

In [ ]:
def select_slices(mask_volume):
    lesion_pixels = mask_volume.sum(axis=(0, 1))
    return np.flatnonzero(lesion_pixels) # select slices with at least one wmh pixel


def build_dataframe(data, slice_indices):
    frames = []

    # for each selected slice, calculate the normalized features and take the corresponding WMH mask as label.
    for slice_idx in slice_indices:
        features = prepare_slice_features(data, slice_idx)
        mask = data["mask"][:, :, slice_idx].astype(np.uint8)

        frames.append(
            pd.DataFrame(
                {
                    "slice_idx": slice_idx,
                    "intensity": features["intensity"].ravel(),
                    "mean": features["mean"].ravel(),
                    "std": features["std"].ravel(),
                    "skewness": features["skew"].ravel(),
                    "kurtosis": features["kurtosis"].ravel(),
                    "contrast": features["contrast"].ravel(),
                    "range": features["range"].ravel(),
                    "x": features["x"].ravel(),
                    "y": features["y"].ravel(),
                    "wmh": mask.ravel(),
                }
            )
        )

    return pd.concat(frames, ignore_index=True)


data = load_dataset(data_dir)
slice_indices = select_slices(data["mask"])
df = build_dataframe(data, slice_indices)


print(f"Number of selected slices: {len(slice_indices)}")

print("Feature summary:")
display(df[feature_columns + ["wmh"]].agg(["min", "max", "mean"]))

print("Random pixels with at least one non-zero feature:")
non_background = df[df[feature_columns].sum(axis=1) > 0]
display(non_background.sample(10, random_state=42))

print("Random WMH pixels:")
display(df[df["wmh"] == 1].sample(10, random_state=42))


### Split and balance training set

The split is done by slice. Only the training set is downsampled to reduce class imbalance.

In [ ]:
slices = df["slice_idx"].unique()
train_slices, val_test_slices = train_test_split(
    slices,
    test_size=0.30,
    random_state=42,
)
val_slices, test_slices = train_test_split(
    val_test_slices,
    test_size=0.50,
    random_state=42,
)

train_df = df[df["slice_idx"].isin(train_slices)].copy()
val_df = df[df["slice_idx"].isin(val_slices)].copy()
test_df = df[df["slice_idx"].isin(test_slices)].copy()

positives = train_df[train_df["wmh"] == 1] 
negatives = train_df[train_df["wmh"] == 0]
sampled_negatives = negatives.sample( # downsample of negative pixels to balance the dataset
    n=min(len(negatives), len(positives)*4), # max 4 negative pixels per positive pixel
    random_state=42,
)
train_balanced_df = pd.concat([positives, sampled_negatives], ignore_index=True) # dataset balanced
train_balanced_df = train_balanced_df.sample(frac=1, random_state=42) # shuffle the balanced dataset

### Scale features and train of the Neural Network

In [ ]:
from copy import deepcopy
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
import warnings
import numpy as np

def cross_entropy_loss(model, X, y):
    proba = model.predict_proba(X)
    proba = np.clip(proba, 1e-10, 1 - 1e-10)
    loss = -np.mean(y * np.log(proba[:, 1]) + (1 - y) * np.log(proba[:, 0]))
    return loss

def dice_score(y_true, y_pred):
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    denom = y_true.sum() + y_pred.sum()
    return 1.0 if denom == 0 else 2 * np.logical_and(y_true, y_pred).sum() / denom

X_train = train_balanced_df[feature_columns].to_numpy()
y_train = train_balanced_df["wmh"].to_numpy()
X_val = val_df[feature_columns].to_numpy()
y_val_np = val_df["wmh"].to_numpy()

# ixed sample of validation for threshold search (faster)
rng = np.random.default_rng(42)
val_sample_idx = rng.choice(len(y_val_np), size=min(50000, len(y_val_np)), replace=False)
X_val_sample = X_val[val_sample_idx]
y_val_sample = y_val_np[val_sample_idx]

train_losses = []
val_losses = []
val_dice_scores = []

model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=512,
    learning_rate_init=1e-3,
    max_iter=1,
    warm_start=True,
    random_state=42,
    verbose=False,
)

n_epochs = 100
patience = 10
min_delta = 1e-4
no_improve = 0
best_val_loss = np.inf
best_model = None


from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)


for epoch in range(n_epochs):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        model.fit(X_train, y_train, sample_weight=sample_weights)

    tl = cross_entropy_loss(model, X_train, y_train)
    vl = cross_entropy_loss(model, X_val, y_val_np)

    # search for best threshold on the validation sample to maximize dice score
    val_probs_sample = model.predict_proba(X_val_sample)[:, 1]
    best_epoch_dice = -np.inf
    best_epoch_threshold = 0.5
    for thr in np.linspace(0.01, 0.99, 99):
        y_pred_thr = (val_probs_sample >= thr).astype(np.uint8)
        d = dice_score(y_val_sample, y_pred_thr)
        if d > best_epoch_dice:
            best_epoch_dice = d
            best_epoch_threshold = thr

    train_losses.append(tl)
    val_losses.append(vl)
    val_dice_scores.append(best_epoch_dice)

    print(f"Epoch {epoch+1:3d} | Train loss: {tl:.4f} | Val loss: {vl:.4f} | Val Dice: {best_epoch_dice:.4f} (thr={best_epoch_threshold:.2f})")

    if vl < best_val_loss - min_delta:
        best_val_loss = vl
        best_model = deepcopy(model)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

if best_model is not None:
    model = best_model
    print(f"Restored best model with validation loss: {best_val_loss:.4f}")

In [ ]:

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curves")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_dice_scores, label="Validation Dice", color="green")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Validation Dice")
plt.legend()

plt.tight_layout()
plt.savefig(output_dir / "learning_curves.png", dpi=150)
plt.show()

In [ ]:
# save best model
import joblib
joblib.dump(model, output_dir / "best_model.pkl")
model = joblib.load(output_dir / "best_model.pkl")


### Evaluate the model

The threshold is selected on the validation set by maximizing Dice score.

In [ ]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return 0.0 if tn + fp == 0 else tn / (tn + fp)


def predict(model, df, threshold):
    X = df[feature_columns].to_numpy()
    probabilities = model.predict_proba(X)[:, 1]
    return (probabilities >= threshold).astype(np.uint8)


X_val = val_df[feature_columns].to_numpy()
y_val = val_df["wmh"].to_numpy()
val_probabilities = model.predict_proba(X_val)[:, 1]

threshold_values = np.linspace(0.01, 0.99, 99)
threshold_dice_scores = []
best_threshold = 0.5
best_dice = -np.inf

for threshold in threshold_values:
    y_pred = (val_probabilities >= threshold).astype(np.uint8)
    current_dice = dice_score(y_val, y_pred)
    threshold_dice_scores.append(current_dice)
    if current_dice > best_dice:
        best_dice = current_dice
        best_threshold = threshold

print(f"Best validation threshold: {best_threshold:.2f} | Dice: {best_dice:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(threshold_values, threshold_dice_scores)
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold = {best_threshold:.2f}")
plt.scatter([best_threshold], [best_dice], color="red")
plt.xlabel("Threshold")
plt.ylabel("Dice")
plt.title("Validation Dice vs Threshold")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "dice_vs_threshold.png", dpi=150)
plt.show()


### Metrics and Confusion Matrices

In [ ]:
results = []
sets = {"train": train_df, "validation": val_df, "test": test_df}

for name, set_df in sets.items():
    y_true = set_df["wmh"].to_numpy()
    y_pred = predict(model, set_df, best_threshold)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    results.append(
        {
            "set": name,
            "threshold": best_threshold,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "dice": dice_score(y_true, y_pred),
        }
    )

    ConfusionMatrixDisplay(cm, display_labels=["non-WMH", "WMH"]).plot(
        values_format="d",
        cmap="Blues",
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.savefig(output_dir / f"confusion_matrix_{name}.png", dpi=150)
    plt.show()

results_df = pd.DataFrame(results)
results_df.to_csv(output_dir / "metrics.csv", index=False)
results_df